# Reproduce paper figures and key results
## *The scale-spanning cortical geometry of language*

This is the canonical **manuscript-level reproduction notebook**. It regenerates the
main numerical figures and headline summaries from final precomputed outputs in
`pang_out/`. It does not rerun raw fMRI preprocessing, language-model inference, or
every upstream GLM. Figure 1 is conceptual and is not regenerated here.

See repository-root `PRECOMPUTED_OUTPUTS.md` for the dependency/provenance map.


## 0. Repository setup


In [ ]:
from pathlib import Path
import os

def find_repo_root():
    env = os.environ.get("EIGENMODE_REPO")
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pang_out").exists():
            return p
        raise FileNotFoundError(f"EIGENMODE_REPO points to {p}, but pang_out is absent.")

    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pang_out").exists() and ((p / "code").exists() or (p / "notebooks").exists()):
            return p

    historical = Path.home() / "eigenmode_fingerprints"
    if (historical / "pang_out").exists():
        return historical

    raise FileNotFoundError(
        "Repository not found. Run from eigenmode_fingerprints (or a subdirectory), "
        "or set EIGENMODE_REPO=/absolute/path/to/eigenmode_fingerprints."
    )

REPO_ROOT = find_repo_root()
PANG = REPO_ROOT / "pang_out"
print("Repository root:", REPO_ROOT)


## 0.1 Validate canonical inputs
The check below is intentionally strict. Missing final inputs raise an error rather
than causing the notebook to search for older alternatives in `pang_out/`.


In [ ]:
REQUIRED = {
    "Figure 2": [
        PANG / "group/energy_spectrum_group.csv",
        PANG / "group/energy_spectrum_subject.csv",
        PANG / "spatial_smoothing_null/expected_smoothing_spectra_grid.csv",
        PANG / "mesh_spatial_smoothing_null/mesh_smoothing_spectra.csv",
        PANG / "mesh_spatial_smoothing_null/mesh_smoothing_comparison.csv",
        PANG / "spectral_model_comparison/group_model_comparison_by_range.csv",
        PANG / "spectral_model_comparison/participant_model_comparison_by_range.csv",
        PANG / "spectral_model_comparison/powerlaw_vs_exponential_range_robustness.csv",
        PANG / "spectral_model_comparison/participant_winner_counts_by_range.csv",
    ],
    "Figure 3": [
        PANG / "group_sentence_controls_exact_8pred/sentence_controls_8pred_standardized_subject_level.csv",
        PANG / "group_sentence_controls_exact_8pred/coherent_sentence_bundle_null_8pred_test/coherent_sentence_bundle_null_8pred_10000.csv",
        PANG / "group_sentence_controls_exact_8pred/coherent_sentence_bundle_null_8pred_test/coherent_sentence_bundle_null_8pred_10000_summary.csv",
        PANG / "group_sentence_controls_exact_8pred/sentence_shift_shuffle_null/sentence_shift_shuffle_null_1000.csv",
        PANG / "group_sentence_controls_exact_8pred/sentence_shift_shuffle_null/sentence_shift_shuffle_null_10000_summary.csv",
        PANG / "token_upstream_hrf_temporal_null_test/token_upstream_hrf_null_10000.csv",
        PANG / "token_upstream_hrf_temporal_null_test/token_upstream_hrf_null_10000_summary.csv",
        PANG / "group/energy_spectrum_group.csv",
        PANG / "group/energy_spectrum_subject.csv",
    ],
    "Figure 4": [
        PANG / "validated_linguistic_effects_geometry/validated_four_beta_star_profiles.csv",
        PANG / "validated_linguistic_effects_geometry/validated_four_shapePCA_component_weights.csv",
        PANG / "validated_linguistic_effects_geometry/validated_four_shapePCA_explained_variance.csv",
    ],
    "Figure 5": [
        PANG / "validated_effects_reconstruction/validated_four_effect_reconstruction_diagnostics.csv",
    ] + [
        PANG / f"validated_effects_reconstruction/{effect}/{effect}_hemi-{hemi}_{kind}.npy"
        for effect in ["sentence_onset", "sentence_shift", "token_surprisal", "token_curvature"]
        for hemi in ["L", "R"]
        for kind in ["empirical", "low", "residual"]
    ],
    "Figure 6": [
        PANG / "subcortex/validated_effects_20260921/hippocampal_mean_signal/hipp_mean_validated_effects_participant_bilateral.csv",
        PANG / "subcortex/validated_effects_20260921/hippocampal_eigenspace_language/hc_eigenspace_bilateral_participant.csv",
        PANG / "subcortex/validated_effects_20260921/vta_mean_signal/vta_bilateral_participant.csv",
        PANG / "subcortex/validated_effects_20260921/vta_hipp_coupling/vta_hc_coupling_subject_bilateral.csv",
        PANG / "subcortex/validated_effects_20260921/vta_hipp_sentence_cofluctuation/vta_hc_cofluctuation_bilateral_participant.csv",
        PANG / "subcortex/validated_effects_20260921/vta_cortical_sentence_cofluctuation/vta_cortical_sentence_cofluctuation_participant_bilateral.csv",
        PANG / "subcortex/validated_effects_20260921/vta_cortical_sentence_cofluctuation/sentence_modulation_LOO_paired_participants.csv",
        PANG / "subcortex/validated_effects_20260921/vta_cortical_sentence_cofluctuation/sentence_modulation_LOO_inference.csv",
    ],
}

missing = {fig:[p for p in paths if not p.exists()] for fig,paths in REQUIRED.items()}
missing = {fig:paths for fig,paths in missing.items() if paths}
if missing:
    msg = ["Missing canonical manuscript inputs:"]
    for fig, paths in missing.items():
        msg += ["", fig + ":"] + ["  - " + str(p.relative_to(REPO_ROOT)) for p in paths]
    raise FileNotFoundError("\n".join(msg))

print("Canonical input check passed:")
for fig, paths in REQUIRED.items():
    print(f"  {fig}: {len(paths)} required files present")


## Figure 2 — Empirical cortical modal-energy spectrum during narrative listening
The eigenmode **basis** is intrinsic to cortical geometry; the modal-energy spectrum
shown here is empirical and is estimated from BOLD activity during narrative listening.


In [ ]:
# ============================================================
# FINAL PUBLICATION EXPORT
#
# Creates:
#   Figure 2 A-B
#   Supplementary Figure A-D
#   Supplementary Table 1: candidate spectral models
#   Supplementary Table 2: fitting-range robustness
#
# IMPORTANT:
#   When this notebook is run top-to-bottom, the mesh-null section
#   above creates `spectra` and permanently saves the spectra CSV.
#   On later runs, this export cell can load the saved CSV directly.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import linregress

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = REPO_ROOT

GRP = BASE / "pang_out" / "group"

NULL_ANALYTIC = (
    BASE / "pang_out" / "spatial_smoothing_null"
)

NULL_MESH = (
    BASE / "pang_out" / "mesh_spatial_smoothing_null"
)

MODEL_DIR = (
    BASE / "pang_out" / "spectral_model_comparison"
)

PUB = (
    BASE / "pang_out" / "paper_figures" / "spatial_scaling_revision"
)

PUB.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# CHECK MESH NULL
# ------------------------------------------------------------

MESH_SPECTRA_FILE = (
    NULL_MESH / "mesh_smoothing_spectra.csv"
)

MESH_COMPARISON_FILE = (
    NULL_MESH / "mesh_smoothing_comparison.csv"
)

if "spectra" in globals():

    # Save explicit mesh spectra permanently so we never
    # need to recompute them just for plotting.

    mesh_rows = []

    for fwhm, Earr in sorted(
        spectra.items(),
        key=lambda z: float(z[0])
    ):

        for k, val in enumerate(
            np.asarray(Earr, float)
        ):

            mesh_rows.append({
                "fwhm_mm": float(fwhm),
                "mode_k": int(k),
                "E": float(val)
            })

    mesh_long = pd.DataFrame(
        mesh_rows
    )

    mesh_long.to_csv(
        MESH_SPECTRA_FILE,
        index=False
    )

    # Also save the coarse FWHM comparison if available
    if "comparison" in globals():

        comparison.to_csv(
            MESH_COMPARISON_FILE,
            index=False
        )

    print(
        "Saved in-memory mesh spectra to:",
        MESH_SPECTRA_FILE
    )

elif MESH_SPECTRA_FILE.exists():

    mesh_long = pd.read_csv(
        MESH_SPECTRA_FILE
    )

    print(
        "Loaded saved mesh spectra:",
        MESH_SPECTRA_FILE
    )

else:

    raise RuntimeError(
        "\nExplicit mesh spectra are not in memory and have "
        "not yet been saved to disk.\n\n"
        "Please run the mesh-based spatial-smoothing section above once, then rerun "
        "this cell. This cell will save them permanently."
    )


# ============================================================
# LOAD EMPIRICAL SPECTRUM
# ============================================================

emp = pd.read_csv(
    GRP / "energy_spectrum_group.csv"
)

mode = emp["mode_k"].to_numpy(int)
lam_all = emp["lam"].to_numpy(float)
E_all = emp["Emean"].to_numpy(float)
SEM_all = emp["Esem"].to_numpy(float)

keep = (
    (mode >= 1)
    & (lam_all > 0)
    & (E_all > 0)
)

mode1 = mode[keep]
lam1 = lam_all[keep]
Eemp = E_all[keep]
SEMemp = SEM_all[keep]

# primary fit
fitmask = (
    (mode1 >= 1)
    & (mode1 <= 60)
)

fit = linregress(
    np.log10(lam1[fitmask]),
    np.log10(Eemp[fitmask])
)

alpha = fit.slope
r2 = fit.rvalue**2

Efit = (
    10**fit.intercept
    * lam1[fitmask]**fit.slope
)


def normalize(E):
    E = np.asarray(E, float)
    return E / np.sum(E)


# ============================================================
# LOAD BEST EXPLICIT MESH NULL
# ============================================================

if MESH_COMPARISON_FILE.exists():

    mesh_comp = pd.read_csv(
        MESH_COMPARISON_FILE
    )

else:

    # reconstruct comparison from known spectra

    rows = []

    for fwhm, d in mesh_long.groupby(
        "fwhm_mm"
    ):

        d = d.sort_values("mode_k")

        Enull = (
            d.loc[
                d["mode_k"] >= 1,
                "E"
            ].to_numpy(float)
        )

        p1 = normalize(Eemp)
        p2 = normalize(Enull)

        mse = np.mean(
            (
                np.log10(p1 + 1e-15)
                - np.log10(p2 + 1e-15)
            )**2
        )

        rows.append({
            "fwhm_mm": fwhm,
            "log_mse": mse
        })

    mesh_comp = pd.DataFrame(rows)

    mesh_comp.to_csv(
        MESH_COMPARISON_FILE,
        index=False
    )


best_mesh_row = mesh_comp.loc[
    mesh_comp["log_mse"].idxmin()
]

BEST_MESH_FWHM = float(
    best_mesh_row["fwhm_mm"]
)

BEST_MESH_MSE = float(
    best_mesh_row["log_mse"]
)

mesh_best = (
    mesh_long[
        mesh_long["fwhm_mm"]
        == BEST_MESH_FWHM
    ]
    .sort_values("mode_k")
)

Emesh = (
    mesh_best.loc[
        mesh_best["mode_k"] >= 1,
        "E"
    ]
    .to_numpy(float)
)

print(
    f"Best explicit mesh null: "
    f"{BEST_MESH_FWHM:.1f} mm, "
    f"log-MSE={BEST_MESH_MSE:.6f}"
)


# ============================================================
# LOAD ANALYTICAL NULL
# ============================================================

analytic_grid_file = (
    NULL_ANALYTIC
    / "expected_smoothing_spectra_grid.csv"
)

smooth = pd.read_csv(
    analytic_grid_file
)

best_analytic = smooth.loc[
    smooth[
        "spectral_log_mse_1_119"
    ].idxmin()
]

BEST_ANALYTIC_FWHM = float(
    best_analytic["fwhm_mm"]
)

BEST_ANALYTIC_MSE = float(
    best_analytic[
        "spectral_log_mse_1_119"
    ]
)

tau_analytic = (
    BEST_ANALYTIC_FWHM**2
    / (16.0 * np.log(2.0))
)

Eanalytic = np.exp(
    -2.0 * tau_analytic * lam1
)

print(
    f"Best analytical null: "
    f"{BEST_ANALYTIC_FWHM:.1f} mm, "
    f"log-MSE={BEST_ANALYTIC_MSE:.6f}"
)


# ============================================================
# LOAD MODEL COMPARISON RESULTS
# ============================================================

group_models = pd.read_csv(
    MODEL_DIR
    / "group_model_comparison_by_range.csv"
)

participant_models = pd.read_csv(
    MODEL_DIR
    / "participant_model_comparison_by_range.csv"
)

robustness = pd.read_csv(
    MODEL_DIR
    / "powerlaw_vs_exponential_range_robustness.csv"
)

winner_counts = pd.read_csv(
    MODEL_DIR
    / "participant_winner_counts_by_range.csv"
)


# ============================================================
# FIGURE 2 — MAIN TEXT
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.6)
)

# ------------------------------------------------------------
# PANEL A — empirical spectrum + primary power-law fit
# ------------------------------------------------------------

ax = axes[0]

ax.plot(
    lam1,
    Eemp,
    marker="o",
    markersize=3,
    linewidth=1.0,
    label="Empirical"
)

# SEM envelope
lower = np.maximum(
    Eemp - SEMemp,
    1e-15
)

upper = Eemp + SEMemp

ax.fill_between(
    lam1,
    lower,
    upper,
    alpha=0.2
)

ax.plot(
    lam1[fitmask],
    Efit,
    linestyle="--",
    linewidth=2,
    label=(
        rf"Power law, $\alpha={alpha:.2f}$"
        "\n"
        rf"$R^2={r2:.2f}$"
    )
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(
    r"Laplace–Beltrami eigenvalue $\lambda$"
)

ax.set_ylabel(
    "Modal energy"
)

ax.legend(
    frameon=False,
    fontsize=9
)

ax.text(
    -0.12,
    1.04,
    "A",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)


# ------------------------------------------------------------
# PANEL B — empirical vs explicit spatial-smoothness null
# ------------------------------------------------------------

ax = axes[1]

ax.plot(
    lam1,
    normalize(Eemp),
    marker="o",
    markersize=3,
    linewidth=1.0,
    label="Empirical"
)

ax.plot(
    lam1,
    normalize(Emesh),
    linewidth=2,
    label=(
        f"Best smooth-field null "
        f"({BEST_MESH_FWHM:.0f} mm)"
    )
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(
    r"Laplace–Beltrami eigenvalue $\lambda$"
)

ax.set_ylabel(
    "Normalized modal energy"
)

ax.legend(
    frameon=False,
    fontsize=9
)

ax.text(
    -0.12,
    1.04,
    "B",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)

fig.tight_layout(
    w_pad=2.2
)

fig.savefig(
    PUB / "figure2_revised_spatial_scaling.png",
    dpi=400,
    bbox_inches="tight"
)

fig.savefig(
    PUB / "figure2_revised_spatial_scaling.pdf",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# SUPPLEMENTARY FIGURE — FOUR PANELS
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(11.5, 9.0)
)

# ------------------------------------------------------------
# A — mesh-null mismatch as a function of smoothing scale
# ------------------------------------------------------------

ax = axes[0, 0]

ax.plot(
    mesh_comp["fwhm_mm"],
    mesh_comp["log_mse"],
    marker="o",
    linewidth=1.5
)

ax.axvline(
    BEST_MESH_FWHM,
    linestyle="--",
    linewidth=1.3
)

ax.set_xlabel(
    "Surface smoothing FWHM (mm)"
)

ax.set_ylabel(
    "Mean squared log spectral error"
)

ax.text(
    -0.12,
    1.04,
    "A",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)


# ------------------------------------------------------------
# B — empirical + mesh null + analytical null
# ------------------------------------------------------------

ax = axes[0, 1]

ax.plot(
    lam1,
    normalize(Eemp),
    marker="o",
    markersize=2.5,
    linewidth=1,
    label="Empirical"
)

ax.plot(
    lam1,
    normalize(Emesh),
    linewidth=2,
    label=f"Mesh null ({BEST_MESH_FWHM:.0f} mm)"
)

ax.plot(
    lam1,
    normalize(Eanalytic),
    linestyle="--",
    linewidth=2,
    label=f"Analytical null ({BEST_ANALYTIC_FWHM:.0f} mm)"
)

ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(
    r"Laplace–Beltrami eigenvalue $\lambda$"
)

ax.set_ylabel(
    "Normalized modal energy"
)

ax.legend(
    frameon=False,
    fontsize=8
)

ax.text(
    -0.12,
    1.04,
    "B",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)


# ------------------------------------------------------------
# C — competing spectral models, primary k=1-60
# ------------------------------------------------------------

ax = axes[1, 0]

primary = group_models[
    (group_models["kmin"] == 1)
    & (group_models["kmax"] == 60)
].copy()

lam60 = lam1[fitmask]
E60 = Eemp[fitmask]

ax.scatter(
    lam60,
    E60,
    s=14,
    label="Empirical"
)

# Generate model predictions directly from saved parameters
for _, row in primary.iterrows():

    name = row["model"]

    if name == "Power law":

        # refit normalization c only, with saved alpha
        a = float(row["alpha"])

        c = np.mean(
            np.log(E60)
            - a * np.log(lam60)
        )

        pred = np.exp(c) * lam60**a


    elif name == "Exponential":

        b = float(row["b"])

        c = np.mean(
            np.log(E60)
            + b * lam60
        )

        pred = np.exp(
            c - b * lam60
        )


    elif name == "Stretched exponential":

        b = float(row["b"])
        gamma = float(row["gamma"])

        c = np.mean(
            np.log(E60)
            + b * lam60**gamma
        )

        pred = np.exp(
            c - b * lam60**gamma
        )


    elif name == "Power law + cutoff":

        a = float(row["alpha"])
        lc = float(row["lambda_c"])

        c = np.mean(
            np.log(E60)
            - a * np.log(lam60)
            + lam60 / lc
        )

        pred = (
            np.exp(c)
            * lam60**a
            * np.exp(-lam60 / lc)
        )

    else:
        continue

    order = np.argsort(lam60)

    ax.plot(
        lam60[order],
        pred[order],
        linewidth=1.5,
        label=name
    )


ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlabel(
    r"Laplace–Beltrami eigenvalue $\lambda$"
)

ax.set_ylabel(
    "Modal energy"
)

ax.legend(
    frameon=False,
    fontsize=7.5
)

ax.text(
    -0.12,
    1.04,
    "C",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)


# ------------------------------------------------------------
# D — participant ΔAICc robustness
# ------------------------------------------------------------

ax = axes[1, 1]

range_order = [
    (1, 40),
    (1, 60),
    (10, 80)
]

range_labels = [
    "1–40",
    "1–60",
    "10–80"
]

for xpos, (kmin, kmax) in enumerate(
    range_order
):

    d = participant_models[
        (participant_models["kmin"] == kmin)
        & (participant_models["kmax"] == kmax)
    ]

    wide = d.pivot(
        index="subject",
        columns="model",
        values="AICc"
    )

    delta = (
        wide["Exponential"]
        - wide["Power law"]
    ).to_numpy(float)

    # deterministic horizontal jitter purely for visibility
    jitter = np.linspace(
        -0.12,
        0.12,
        len(delta)
    )

    ax.scatter(
        np.full(len(delta), xpos)
        + jitter,
        delta,
        s=18,
        alpha=0.75
    )

    ax.plot(
        [xpos - 0.20, xpos + 0.20],
        [np.median(delta)] * 2,
        linewidth=3
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)

ax.set_xticks(
    range(len(range_labels))
)

ax.set_xticklabels(
    range_labels
)

ax.set_xlabel(
    "Mode fitting range"
)

ax.set_ylabel(
    r"$\Delta$AICc (Exponential − Power law)"
)

ax.text(
    -0.12,
    1.04,
    "D",
    transform=ax.transAxes,
    fontsize=15,
    fontweight="bold"
)


fig.tight_layout(
    h_pad=2.3,
    w_pad=2.2
)

fig.savefig(
    PUB / "supplementary_spatial_scaling_robustness.png",
    dpi=400,
    bbox_inches="tight"
)

fig.savefig(
    PUB / "supplementary_spatial_scaling_robustness.pdf",
    bbox_inches="tight"
)

plt.show()


# ============================================================
# SUPPLEMENTARY TABLE 1
# PRIMARY CANDIDATE SPECTRAL MODELS, k=1-60
# ============================================================

table1 = primary.copy()

table1 = table1[
    [
        "model",
        "R2_log",
        "AICc",
        "delta_AICc",
        "Akaike_weight",
        "BIC",
        "delta_BIC",
        "alpha",
        "b",
        "gamma",
        "lambda_c"
    ]
].sort_values(
    "AICc"
)

table1 = table1.rename(
    columns={
        "model": "Model",
        "R2_log": "R2_log",
        "AICc": "AICc",
        "delta_AICc": "Delta_AICc",
        "Akaike_weight": "Akaike_weight",
        "BIC": "BIC",
        "delta_BIC": "Delta_BIC",
        "alpha": "alpha",
        "b": "b",
        "gamma": "gamma",
        "lambda_c": "lambda_c"
    }
)

table1.to_csv(
    PUB / "supplementary_table_1_candidate_models.csv",
    index=False
)

print("\n")
print("=" * 80)
print("SUPPLEMENTARY TABLE 1")
print("Candidate spectral models, primary range k=1-60")
print("=" * 80)

display(
    table1.round({
        "R2_log": 3,
        "AICc": 2,
        "Delta_AICc": 2,
        "Akaike_weight": 3,
        "BIC": 2,
        "Delta_BIC": 2,
        "alpha": 3,
        "b": 3,
        "gamma": 3,
        "lambda_c": 4
    })
)


# ============================================================
# SUPPLEMENTARY TABLE 2
# FITTING-RANGE ROBUSTNESS
# ============================================================

winner_lookup = (
    winner_counts[
        winner_counts["model"]
        == "Power law"
    ]
    .set_index("range")[
        "n_winners"
    ]
    .to_dict()
)

table2_rows = []

for _, row in robustness.iterrows():

    rg = str(row["range"])

    # handle pandas potentially interpreting range strangely
    if rg.endswith(".0"):
        rg = rg[:-2]

    table2_rows.append({
        "Mode range": rg,
        "Group Delta_AICc Exp-PL":
            np.nan,  # filled below
        "Median participant Delta_AICc":
            row["median_delta_AICc_Exp_minus_PL"],
        "PL preferred to Exp":
            f"{int(row['n_favour_PL'])}/{int(row['n'])}",
        "Delta_AICc > 10":
            f"{int(row['n_strong_PL_delta_gt_10'])}/{int(row['n'])}",
        "PL overall AICc winner":
            f"{int(winner_lookup.get(rg, np.nan))}/{int(row['n'])}"
            if rg in winner_lookup else ""
    })


table2 = pd.DataFrame(
    table2_rows
)

# Add group Exp-vs-PL ΔAICc directly
for i, (kmin, kmax) in enumerate(
    range_order
):

    d = group_models[
        (group_models["kmin"] == kmin)
        & (group_models["kmax"] == kmax)
    ]

    aicc_exp = float(
        d.loc[
            d["model"] == "Exponential",
            "AICc"
        ].iloc[0]
    )

    aicc_pl = float(
        d.loc[
            d["model"] == "Power law",
            "AICc"
        ].iloc[0]
    )

    table2.loc[
        i,
        "Group Delta_AICc Exp-PL"
    ] = (
        aicc_exp - aicc_pl
    )


table2.to_csv(
    PUB / "supplementary_table_2_range_robustness.csv",
    index=False
)

print("\n")
print("=" * 80)
print("SUPPLEMENTARY TABLE 2")
print("Robustness to fitted mode range")
print("=" * 80)

display(
    table2.round({
        "Group Delta_AICc Exp-PL": 2,
        "Median participant Delta_AICc": 2
    })
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("EXPORT COMPLETE")
print("=" * 80)

print("\nMain Figure 2:")
print(
    PUB / "figure2_revised_spatial_scaling.png"
)
print(
    PUB / "figure2_revised_spatial_scaling.pdf"
)

print("\nSupplementary Figure:")
print(
    PUB / "supplementary_spatial_scaling_robustness.png"
)
print(
    PUB / "supplementary_spatial_scaling_robustness.pdf"
)

print("\nTables:")
print(
    PUB / "supplementary_table_1_candidate_models.csv"
)
print(
    PUB / "supplementary_table_2_range_robustness.csv"
)

print("\nPermanent mesh-spectrum files:")
print(MESH_SPECTRA_FILE)
print(MESH_COMPARISON_FILE)


## Figure 3 — Stimulus-specific effects preferentially concentrate in long-wavelength modes
Panels A–B validate sentence- and token-level effects. Panels C–D then retain only
sentence onset, sentence shift, token surprisal, and token curvature. The reference
in panel D is the empirical cortical modal-energy distribution during narrative
listening. Historical filenames containing `intrinsic` are retained only for backward
compatibility.


In [ ]:
# ============================================================
# FINAL INTEGRATED FIGURE — REVISED SPACING VERSION
#
# A. Standardized partial beta* profiles of four validated effects
# B. Shape-PCA loading space of those four effects
# C. Sentence-level validation nulls
# D. Token-level upstream-of-HRF validation nulls
#
# Logic:
#   controlled estimation
#       -> stimulus-specific validation
#       -> retain validated linguistic effects
#       -> characterize their eigenmode geometry
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.gridspec import (
    GridSpec,
    GridSpecFromSubplotSpec,
)

from scipy.stats import gaussian_kde


# ============================================================
# PATHS
# ============================================================

BASE = REPO_ROOT
PANG = BASE / "pang_out"

GEOM = (
    PANG
    / "validated_linguistic_effects_geometry"
)

SENT_NULL_DIR = (
    PANG
    / "group_sentence_controls_exact_8pred"
    / "coherent_sentence_bundle_null_8pred_test"
)

SHIFT_NULL_DIR = (
    PANG
    / "group_sentence_controls_exact_8pred"
    / "sentence_shift_shuffle_null"
)

TOKEN_NULL_DIR = (
    PANG
    / "token_upstream_hrf_temporal_null_test"
)

FIGDIR = (
    PANG
    / "paper_figures"
)

FIGDIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD VALIDATED FOUR-PROFILE GEOMETRY
# ============================================================

profiles = pd.read_csv(
    GEOM
    / "validated_four_beta_star_profiles.csv",
    index_col=0,
)

profiles.index = profiles.index.astype(int)


weights = pd.read_csv(
    GEOM
    / "validated_four_shapePCA_component_weights.csv",
    index_col=0,
)


ev = pd.read_csv(
    GEOM
    / "validated_four_shapePCA_explained_variance.csv"
)


pc1_pct = float(
    ev.loc[
        ev["PC"] == "PC1",
        "explained_variance_percent",
    ].iloc[0]
)

pc2_pct = float(
    ev.loc[
        ev["PC"] == "PC2",
        "explained_variance_percent",
    ].iloc[0]
)

pc12_pct = (
    pc1_pct
    + pc2_pct
)


# ============================================================
# LOAD SENTENCE NULLS
# ============================================================

sentence_null = pd.read_csv(
    SENT_NULL_DIR
    / "coherent_sentence_bundle_null_8pred_10000.csv"
)

sentence_null_summary = pd.read_csv(
    SENT_NULL_DIR
    / "coherent_sentence_bundle_null_8pred_10000_summary.csv"
)


# Full saved distribution = 1,000 permutations.
# Definitive inferential summary = 10,000 permutations.
shift_null = pd.read_csv(
    SHIFT_NULL_DIR
    / "sentence_shift_shuffle_null_1000.csv"
)

shift_null_summary = pd.read_csv(
    SHIFT_NULL_DIR
    / "sentence_shift_shuffle_null_10000_summary.csv"
)


# ============================================================
# LOAD TOKEN NULL
# ============================================================

token_null = pd.read_csv(
    TOKEN_NULL_DIR
    / "token_upstream_hrf_null_10000.csv"
)

token_summary = pd.read_csv(
    TOKEN_NULL_DIR
    / "token_upstream_hrf_null_10000_summary.csv"
)


# ============================================================
# COLORS
# ============================================================

COLORS = {
    "Sentence onset": "#0072B2",
    "Sentence shift": "#D55E00",
    "Token surprisal": "#009E73",
    "Token curvature": "#CC79A7",
}

FAILED = "#9A9A9A"
NULL_COLOR = "#B8B8B8"


# ============================================================
# HELPER: IDENTIFY COLUMNS
# ============================================================

def find_col(
    df,
    include,
    exclude=None,
):

    if isinstance(
        include,
        str,
    ):
        include = [include]

    include = [
        x.lower()
        for x in include
    ]

    if exclude is None:
        exclude = []

    if isinstance(
        exclude,
        str,
    ):
        exclude = [exclude]

    exclude = [
        x.lower()
        for x in exclude
    ]

    matches = []

    for col in df.columns:

        low = col.lower()

        if (
            all(
                x in low
                for x in include
            )
            and not any(
                x in low
                for x in exclude
            )
        ):
            matches.append(col)

    if len(matches) != 1:

        raise RuntimeError(
            "Could not uniquely identify column.\n"
            f"include={include}\n"
            f"exclude={exclude}\n"
            f"matches={matches}\n"
            f"columns={df.columns.tolist()}"
        )

    return matches[0]


# ============================================================
# SENTENCE NULL COLUMNS
# ============================================================

onset_null_col = find_col(
    sentence_null,
    [
        "onset",
        "rms",
    ],
)


shift_null_col = find_col(
    shift_null,
    ["mean"],
    exclude=[
        "abs",
        "seconds",
        "perm",
    ],
)


# ============================================================
# EMPIRICAL SENTENCE VALUES + P
# ============================================================

# ------------------------------------------------------------
# Sentence onset
# ------------------------------------------------------------

onset_summary_rows = (
    sentence_null_summary[
        sentence_null_summary
        .astype(str)
        .apply(
            lambda row:
                row
                .str.lower()
                .str.contains("onset")
                .any(),
            axis=1,
        )
    ]
)


if "metric" in onset_summary_rows.columns:

    tmp = onset_summary_rows[
        onset_summary_rows[
            "metric"
        ]
        .astype(str)
        .str.lower()
        .str.contains("rms")
    ]

    if len(tmp) > 0:
        onset_summary_rows = tmp


if len(onset_summary_rows) != 1:

    display(
        onset_summary_rows
    )

    raise RuntimeError(
        "Onset summary row not unique."
    )


onset_row = (
    onset_summary_rows
    .iloc[0]
)


onset_emp = float(
    onset_row[
        "empirical"
    ]
)

onset_p = float(
    onset_row[
        "p_perm_greater"
    ]
)


# ------------------------------------------------------------
# Sentence shift
# ------------------------------------------------------------

shift_summary_rows = (
    shift_null_summary.copy()
)


if "metric" in shift_summary_rows.columns:

    tmp = shift_summary_rows[
        (
            shift_summary_rows[
                "metric"
            ]
            .astype(str)
            .str.lower()
            .str.contains("signed")
        )
        |
        (
            shift_summary_rows[
                "metric"
            ]
            .astype(str)
            .str.lower()
            .eq("mean_beta_star")
        )
    ]

    if len(tmp) > 0:
        shift_summary_rows = tmp


if len(shift_summary_rows) != 1:

    numeric_emp = pd.to_numeric(
        shift_summary_rows[
            "empirical"
        ],
        errors="coerce",
    )

    ii = np.nanargmin(
        np.abs(
            numeric_emp.to_numpy()
            - 0.007421
        )
    )

    shift_row = (
        shift_summary_rows
        .iloc[ii]
    )

else:

    shift_row = (
        shift_summary_rows
        .iloc[0]
    )


shift_emp = float(
    shift_row[
        "empirical"
    ]
)

shift_p = float(
    shift_row[
        "p_perm_greater"
    ]
)


# ============================================================
# TOKEN NULL MAPPING
# ============================================================

TOKEN_MAP = {
    "Token surprisal":
        "qwen_surprisal_profile_rms",

    "Token curvature":
        "curvature_profile_rms",

    "Token shift":
        "token_shift_profile_rms",

    "AR residual":
        "ar_trajectory_residual_profile_rms",

    "Subspace exit":
        "subspace_exit_profile_rms",
}


TOKEN_SUMMARY_MAP = {
    "Token surprisal":
        "Qwen surprisal",

    "Token curvature":
        "Curvature",

    "Token shift":
        "Token shift",

    "AR residual":
        "AR trajectory residual",

    "Subspace exit":
        "Subspace exit",
}


# ============================================================
# PANEL C NULL-DISTRIBUTION HELPER
# ============================================================

def plot_null_distribution(
    ax,
    null_values,
    empirical,
    color,
    p_value,
    title=None,
    xlabel=None,
    bins=45,
    p_x=0.95,
    p_y=0.92,
    p_ha="right",
):

    null_values = np.asarray(
        null_values,
        dtype=float,
    )

    null_values = null_values[
        np.isfinite(
            null_values
        )
    ]


    # --------------------------------------------------------
    # Histogram
    # --------------------------------------------------------

    ax.hist(
        null_values,
        bins=bins,
        density=True,
        color=NULL_COLOR,
        alpha=0.55,
        edgecolor="none",
    )


    # --------------------------------------------------------
    # KDE
    # --------------------------------------------------------

    kde = gaussian_kde(
        null_values
    )


    xmin = min(
        null_values.min(),
        empirical,
    )

    xmax0 = max(
        null_values.max(),
        empirical,
    )

    xr = (
        xmax0
        - xmin
    )

    if xr == 0:
        xr = 1.0


    xmin = (
        xmin
        - 0.05 * xr
    )

    xmax = (
        xmax0
        + 0.08 * xr
    )


    xx = np.linspace(
        xmin,
        xmax,
        500,
    )


    ax.plot(
        xx,
        kde(xx),
        color="0.35",
        lw=1.4,
    )


    # --------------------------------------------------------
    # Empirical statistic
    # --------------------------------------------------------

    ax.axvline(
        empirical,
        color=color,
        lw=2.5,
        zorder=5,
    )


    # --------------------------------------------------------
    # Permutation p
    # --------------------------------------------------------

    if (
        p_value
        <= 1 / 10001 + 1e-10
    ):

        ptxt = (
            r"$p_{\mathrm{perm}}<.0001$"
        )

    elif p_value < .001:

        ptxt = (
            rf"$p_{{\mathrm{{perm}}}}={p_value:.4f}$"
        )

    else:

        ptxt = (
            rf"$p_{{\mathrm{{perm}}}}={p_value:.3f}$"
        )


    ax.text(
        p_x,
        p_y,
        ptxt,
        transform=ax.transAxes,
        ha=p_ha,
        va="top",
        fontsize=9,
        color=color,
    )


    if title is not None:

        ax.set_title(
            title,
            fontsize=10.5,
            fontweight="bold",
            pad=7,
        )


    if xlabel is not None:

        ax.set_xlabel(
            xlabel,
            fontsize=9.5,
        )


    ax.set_ylabel(
        "Density",
        fontsize=9.5,
    )


    ax.spines[
        [
            "top",
            "right",
        ]
    ].set_visible(False)


    ax.tick_params(
        labelsize=8.5
    )




# ============================================================
# RAW-BETA / CONCENTRATION CALCULATIONS FROM FORMER FIGURE 4
# ============================================================

# ============================================================
# FINAL FIGURE
#
# VALIDATED LINGUISTIC EFFECTS PREFERENTIALLY CONCENTRATE
# WITHIN LONG-WAVELENGTH CORTICAL EIGENMODES
#
# Panel A:
#   Raw partial-beta profiles for the four independently
#   validated linguistic effects.
#
# Panel B:
#   Cumulative |raw beta| concentration compared with the
#   exact intrinsic modal-energy spectrum used in Figure 2.
#
# Participant-level paired inference at K=20:
#   linguistic C_beta(20) versus each participant's own
#   intrinsic C_E(20).
#
# Sentence effects:
#   final 8-predictor sentence GLM
#
# Token effects:
#   continuous-context Qwen models controlling for word rate
#
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# PATHS
# ============================================================

BASE = REPO_ROOT
PANG = BASE / "pang_out"

SENT_DIR = (
    PANG
    / "group_sentence_controls_exact_8pred"
)

SURP_DIR = (
    PANG
    / "standardized_beta_profiles_qwen3_0p6b_continuous_centered_wordrate_surprisal"
)

CURV_DIR = (
    PANG
    / "group_qwen3_0p6b_continuous_centered_wordrate_curvature_glm"
)

# Exact intrinsic spectrum used for Figure 2
INTRINSIC_FILE = (
    PANG
    / "group"
    / "energy_spectrum_group.csv"
)

OUTDIR = (
    PANG
    / "validated_linguistic_effects_longwavelength"
)

FIGDIR = (
    PANG
    / "paper_figures"
)

OUTDIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGDIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# DISPLAY SETTINGS
# ============================================================

COLORS = {
    "Sentence onset": "#0072B2",
    "Sentence shift": "#D55E00",
    "Token surprisal": "#009E73",
    "Token curvature": "#CC79A7",
}

PREDICTORS = [
    "Sentence onset",
    "Sentence shift",
    "Token surprisal",
    "Token curvature",
]


# ============================================================
# 1. SENTENCE EFFECTS
#
# Raw partial beta from the FINAL 8-predictor model.
#
# The subject-level table has already averaged over runs.
# We:
#   hemisphere-average within participant
#   -> group-average across participants
# ============================================================

sentence_file = (
    SENT_DIR
    / "sentence_controls_8pred_standardized_subject_level.csv"
)

sent = pd.read_csv(
    sentence_file
)


SENTENCE_LABELS = {
    "sentence_onset": "Sentence onset",
    "sentence_shift": "Sentence shift",
}


sent = sent[
    sent["predictor"].isin(
        SENTENCE_LABELS.keys()
    )
    &
    sent["mode"].between(
        1,
        119,
    )
].copy()


sent["predictor"] = (
    sent["predictor"]
    .map(
        SENTENCE_LABELS
    )
)


# ------------------------------------------------------------
# Hemisphere-average within participant
# ------------------------------------------------------------

sent_subject = (
    sent
    .groupby(
        [
            "subject",
            "mode",
            "predictor",
        ],
        as_index=False,
    )
    .agg(
        beta=(
            "beta",
            "mean",
        )
    )
)


# ------------------------------------------------------------
# Group mean
# ------------------------------------------------------------

sent_group = (
    sent_subject
    .groupby(
        [
            "mode",
            "predictor",
        ],
        as_index=False,
    )
    .agg(
        beta_mean=(
            "beta",
            "mean",
        ),
        beta_sem=(
            "beta",
            lambda x:
                np.std(
                    x,
                    ddof=1,
                )
                / np.sqrt(
                    len(x)
                ),
        ),
        N=(
            "subject",
            "nunique",
        ),
    )
)


# ============================================================
# 2. TOKEN SURPRISAL
#
# Raw beta from final continuous-context Qwen surprisal model
# controlling for word rate.
# ============================================================

surp_file = (
    SURP_DIR
    / "standardized_effects_group_surprisal.csv"
)

surp = pd.read_csv(
    surp_file
)


surp = surp[
    surp["mode_k"].between(
        1,
        119,
    )
].copy()


surp_group = pd.DataFrame(
    {
        "mode":
            surp[
                "mode_k"
            ].astype(int),

        "predictor":
            "Token surprisal",

        "beta_mean":
            surp[
                "beta_raw_mean"
            ].astype(float),

        "beta_sem":
            np.nan,

        "N":
            surp[
                "N_subjects"
            ],
    }
)


# ============================================================
# 3. TOKEN CURVATURE
#
# Raw beta from final continuous-context curvature model
# controlling for word rate.
# ============================================================

curv_L_file = (
    CURV_DIR
    / "group_qwen3_0p6b_continuous_centered_wordrate_curvature_hemi-L_by_mode_subject_level.csv"
)

curv_R_file = (
    CURV_DIR
    / "group_qwen3_0p6b_continuous_centered_wordrate_curvature_hemi-R_by_mode_subject_level.csv"
)


curv_L = pd.read_csv(
    curv_L_file
)

curv_R = pd.read_csv(
    curv_R_file
)


curv = curv_L[
    [
        "mode_k",
        "beta_mean",
        "beta_sem",
        "N_subj",
        "lam",
    ]
].merge(
    curv_R[
        [
            "mode_k",
            "beta_mean",
            "beta_sem",
            "N_subj",
            "lam",
        ]
    ],
    on="mode_k",
    suffixes=(
        "_L",
        "_R",
    ),
    validate="one_to_one",
)


curv = curv[
    curv["mode_k"].between(
        1,
        119,
    )
].copy()


curv_group = pd.DataFrame(
    {
        "mode":
            curv[
                "mode_k"
            ].astype(int),

        "predictor":
            "Token curvature",

        "beta_mean":
            (
                curv[
                    "beta_mean_L"
                ]
                +
                curv[
                    "beta_mean_R"
                ]
            )
            / 2.0,

        "beta_sem":
            np.sqrt(
                curv[
                    "beta_sem_L"
                ] ** 2
                +
                curv[
                    "beta_sem_R"
                ] ** 2
            )
            / 2.0,

        "N":
            np.minimum(
                curv[
                    "N_subj_L"
                ],
                curv[
                    "N_subj_R"
                ],
            ),
    }
)


# ============================================================
# 4. COMBINE FOUR VALIDATED RAW-BETA PROFILES
# ============================================================

raw_long = pd.concat(
    [
        sent_group,
        surp_group,
        curv_group,
    ],
    ignore_index=True,
    sort=False,
)


raw_long = (
    raw_long
    .sort_values(
        [
            "predictor",
            "mode",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# QC
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "RAW PROFILE QC"
)

print(
    "=" * 80
)


for predictor in PREDICTORS:

    d = raw_long[
        raw_long[
            "predictor"
        ] == predictor
    ]

    print(
        f"{predictor:20s} "
        f"N_modes={len(d):3d}  "
        f"mode_range={d['mode'].min()}-{d['mode'].max()}  "
        f"mean_beta={d['beta_mean'].mean(): .6g}  "
        f"mean_abs_beta={np.abs(d['beta_mean']).mean(): .6g}"
    )


# ============================================================
# 5. CUMULATIVE |RAW BETA|
#
# Within each predictor:
#
# C_beta(K) =
#     sum_{k<=K} |beta_k|
#     -------------------
#     sum_{k<=119} |beta_k|
#
# ============================================================

cumulative_rows = []


for predictor in PREDICTORS:

    d = (
        raw_long[
            raw_long[
                "predictor"
            ] == predictor
        ]
        .sort_values(
            "mode"
        )
        .copy()
    )


    abs_beta = np.abs(
        d[
            "beta_mean"
        ].to_numpy(
            float
        )
    )


    cumulative = (
        np.cumsum(
            abs_beta
        )
        /
        np.sum(
            abs_beta
        )
    )


    for mode, beta, ab, cum in zip(
        d["mode"],
        d["beta_mean"],
        abs_beta,
        cumulative,
    ):

        cumulative_rows.append(
            {
                "predictor":
                    predictor,

                "mode":
                    int(mode),

                "beta_mean":
                    float(beta),

                "abs_beta":
                    float(ab),

                "cumulative_abs_beta_fraction":
                    float(cum),
            }
        )


cum_df = pd.DataFrame(
    cumulative_rows
)


# ============================================================
# 6. EXACT FIGURE-2 INTRINSIC MODAL-ENERGY SPECTRUM
# ============================================================

intrinsic = pd.read_csv(
    INTRINSIC_FILE
)


intrinsic = intrinsic[
    intrinsic[
        "mode_k"
    ].between(
        1,
        119,
    )
].copy()


intrinsic = intrinsic.sort_values(
    "mode_k"
)


assert len(
    intrinsic
) == 119


energy = intrinsic[
    "Emean"
].to_numpy(
    float
)


intrinsic[
    "cumulative_energy_fraction"
] = (
    np.cumsum(
        energy
    )
    /
    np.sum(
        energy
    )
)


# ============================================================
# 7. NUMERIC SUMMARY
# ============================================================

K_VALUES = [
    5,
    10,
    20,
    30,
    50,
]


comparison_rows = []


for K in K_VALUES:

    row = {
        "K":
            K,

        "Empirical cortical modal energy":
            100.0
            * float(
                intrinsic.loc[
                    intrinsic[
                        "mode_k"
                    ] == K,
                    "cumulative_energy_fraction",
                ]
                .iloc[0]
            ),
    }


    for predictor in PREDICTORS:

        d = cum_df[
            cum_df[
                "predictor"
            ] == predictor
        ]


        row[
            predictor
        ] = (
            100.0
            * float(
                d.loc[
                    d[
                        "mode"
                    ] == K,
                    "cumulative_abs_beta_fraction",
                ]
                .iloc[0]
            )
        )


    comparison_rows.append(
        row
    )


comparison = pd.DataFrame(
    comparison_rows
)


print(
    "\n" + "=" * 90
)

print(
    "LINGUISTIC MODULATION VS INTRINSIC MODAL ENERGY"
)

print(
    "=" * 90
)

print(
    comparison
    .round(2)
    .to_string(
        index=False
    )
)


# ============================================================
# 8. SAVE NUMERIC DATA
# ============================================================

raw_long.to_csv(
    OUTDIR
    / "raw_beta_profiles_validated.csv",
    index=False,
)


cum_df.to_csv(
    OUTDIR
    / "cumulative_abs_beta_validated.csv",
    index=False,
)


intrinsic[
    [
        "mode_k",
        "lam",
        "Emean",
        "Esem",
        "cumulative_energy_fraction",
    ]
].to_csv(
    OUTDIR
    / "intrinsic_energy_cumulative_figure2.csv",
    index=False,
)


comparison.to_csv(
    OUTDIR
    / "linguistic_vs_intrinsic_lowmode_concentration.csv",
    index=False,
)


# ============================================================
# 9. PARTICIPANT-LEVEL INFERENTIAL RESULTS
#
# These are the results of the paired participant-level test
# already run:
#
# Delta_i =
#   C_beta_i(20) - C_E_i(20)
#
# 100,000 sign-flip permutations.
# BH-FDR across four validated predictors.
#
# We use these only for the concise Panel-B annotation.
# ============================================================

PRIMARY_K = 20

PRIMARY_RESULTS = {
    "Sentence onset": {
        "delta_pp": 9.45,
        "ci_low": 9.07,
        "ci_high": 9.83,
    },

    "Sentence shift": {
        "delta_pp": 7.38,
        "ci_low": 6.77,
        "ci_high": 7.96,
    },

    "Token surprisal": {
        "delta_pp": 10.00,
        "ci_low": 9.60,
        "ci_high": 10.39,
    },

    "Token curvature": {
        "delta_pp": 9.34,
        "ci_low": 8.94,
        "ci_high": 9.76,
    },
}




# ============================================================
# NEW FIGURE 3
# A. Sentence-level stimulus-specific validation
# B. Token-level upstream-of-HRF validation
# C. Raw partial-beta profiles of the four validated effects
# D. Cumulative low-mode concentration vs intrinsic energy
# ============================================================

fig = plt.figure(figsize=(15.2, 10.8))
outer = GridSpec(
    2, 2, figure=fig,
    width_ratios=[1.0, 1.0],
    height_ratios=[0.92, 1.08],
    wspace=0.30, hspace=0.42,
)

# ------------------------- PANEL A -------------------------
gsA = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[0, 0], wspace=0.38)
axA1 = fig.add_subplot(gsA[0, 0])
axA2 = fig.add_subplot(gsA[0, 1])

plot_null_distribution(
    axA1, sentence_null[onset_null_col].to_numpy(float), onset_emp,
    COLORS["Sentence onset"], onset_p,
    title="Sentence onset", xlabel="Profile RMS",
    p_x=0.64, p_y=0.93, p_ha="left",
)
plot_null_distribution(
    axA2, shift_null[shift_null_col].to_numpy(float), shift_emp,
    COLORS["Sentence shift"], shift_p,
    title="Sentence representational shift", xlabel=r"Signed mean $\beta^*$",
    p_x=0.96, p_y=0.93, p_ha="right",
)
axA1.text(-0.25, 1.16, "A", transform=axA1.transAxes,
          fontsize=18, fontweight="bold", va="top")
fig.text(axA1.get_position().x0 + 0.025, axA1.get_position().y1 + 0.025,
         "Sentence-level validation", fontsize=12, fontweight="bold", va="top")

# ------------------------- PANEL B -------------------------
axB = fig.add_subplot(outer[0, 1])
token_order = ["Token surprisal", "Token curvature", "Token shift", "AR residual", "Subspace exit"]
all_vals = [token_null[TOKEN_MAP[n]].to_numpy(float) for n in token_order]
all_concat = np.concatenate(all_vals)
emp_token, p_token = {}, {}
for name in token_order:
    row = token_summary[(token_summary["predictor"] == TOKEN_SUMMARY_MAP[name]) &
                        (token_summary["metric"] == "profile_rms")]
    assert len(row) == 1
    emp_token[name] = float(row["empirical"].iloc[0])
    p_token[name] = float(row["p_perm_greater"].iloc[0])
xmax = max(np.max(all_concat), max(emp_token.values())) * 1.08
xx = np.linspace(0, xmax, 600)
ypos = {name: len(token_order)-1-i for i, name in enumerate(token_order)}
for name in token_order:
    vals = token_null[TOKEN_MAP[name]].to_numpy(float)
    vals = vals[np.isfinite(vals)]
    dens = gaussian_kde(vals)(xx)
    dens = dens / dens.max() * 0.68
    y0 = ypos[name]
    axB.fill_between(xx, y0, y0+dens, color=NULL_COLOR, alpha=0.60, linewidth=0)
    axB.plot(xx, y0+dens, color="0.38", lw=1.0)
    c = COLORS[name] if name in ["Token surprisal", "Token curvature"] else FAILED
    axB.vlines(emp_token[name], y0, y0+0.78, color=c, lw=2.5, zorder=5)
    p = p_token[name]
    ptxt = "p<.0001" if p <= 1/10001 + 1e-10 else (f"p={p:.4f}" if p < .001 else f"p={p:.3f}")
    if name == "Token surprisal":
        px, pha = emp_token[name]-0.0030, "right"
    else:
        px, pha = xmax*0.985, "right"
    axB.text(px, y0+0.30, ptxt, ha=pha, va="center", fontsize=8.7,
             color=c, fontweight="bold" if name in ["Token surprisal", "Token curvature"] else "normal")
axB.set_yticks([ypos[n]+0.22 for n in token_order])
axB.set_yticklabels(token_order, fontsize=9.3)
for tick, name in zip(axB.get_yticklabels(), token_order):
    if name in ["Token surprisal", "Token curvature"]:
        tick.set_color(COLORS[name]); tick.set_fontweight("bold")
    else:
        tick.set_color(FAILED)
axB.set_xlim(0, xmax); axB.set_ylim(-0.2, len(token_order))
axB.set_xlabel("Profile RMS", fontsize=11)
axB.set_title("Token-level validation: upstream-of-HRF temporal null", fontsize=12, fontweight="bold", pad=9)
axB.spines[["top", "right", "left"]].set_visible(False)
axB.tick_params(axis="y", length=0); axB.tick_params(axis="x", labelsize=9.5)
axB.text(-0.12, 1.08, "B", transform=axB.transAxes, fontsize=18, fontweight="bold", va="top")

# ------------------------- PANEL C -------------------------
gsC = GridSpecFromSubplotSpec(1, 4, subplot_spec=outer[1, 0], wspace=0.34)
raw_axes = []
for i, predictor in enumerate(PREDICTORS):
    ax = fig.add_subplot(gsC[0, i]); raw_axes.append(ax)
    d = raw_long[raw_long["predictor"] == predictor].sort_values("mode")
    ax.plot(d["mode"], d["beta_mean"], color=COLORS[predictor], lw=1.8)
    ax.axhline(0, color="0.55", lw=0.8, ls="--")
    ax.set_xlim(1, 119)
    ax.set_title(predictor, fontsize=9.2, fontweight="bold", color=COLORS[predictor], pad=7)
    ax.set_xlabel("$k$", fontsize=9.5)
    if i == 0: ax.set_ylabel("Raw partial β", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=8.0)
raw_axes[0].text(-0.38, 1.12, "C", transform=raw_axes[0].transAxes,
                 fontsize=18, fontweight="bold", va="top")
fig.text(raw_axes[0].get_position().x0 + 0.025, raw_axes[0].get_position().y1 + 0.025,
         "Raw eigenmode profiles of validated effects", fontsize=12, fontweight="bold", va="top")

# ------------------------- PANEL D -------------------------
axD = fig.add_subplot(outer[1, 1])
for predictor in PREDICTORS:
    d = cum_df[cum_df["predictor"] == predictor].sort_values("mode")
    axD.plot(d["mode"], 100*d["cumulative_abs_beta_fraction"],
             color=COLORS[predictor], lw=2.15, label=predictor, zorder=3)
axD.plot(intrinsic["mode_k"], 100*intrinsic["cumulative_energy_fraction"],
         color="black", lw=2.8, ls="--", label="Empirical cortical modal energy", zorder=4)
axD.axvline(PRIMARY_K, color="0.65", lw=1.0, ls="--", zorder=0)
for pct in [50, 90]: axD.axhline(pct, color="0.84", lw=0.8, ls=":", zorder=0)
axD.set_xlim(1, 119); axD.set_ylim(0, 101)
axD.set_xlabel("Number of lowest-order eigenmodes, $K$", fontsize=11)
axD.set_ylabel("Cumulative fraction (%)", fontsize=11)
axD.spines[["top", "right"]].set_visible(False); axD.tick_params(labelsize=9.5)
axD.set_title("Linguistic modulation is more low-mode concentrated than cortical activity",
              fontsize=12, fontweight="bold", pad=9)
axD.text(-0.12, 1.08, "D", transform=axD.transAxes, fontsize=18, fontweight="bold", va="top")
annotation = ("$K=20$: participant-level excess = +7.4 to +10.0 percentage points\n"
              "all paired $p_{perm}<.0001$, $q_{FDR}<.0001$")
axD.text(0.43, 0.31, annotation, transform=axD.transAxes, fontsize=9.0,
         ha="left", va="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="0.75", alpha=0.92))
axD.legend(frameon=False, fontsize=8.8, ncol=2, loc="lower right")

fig.suptitle(
    "Stimulus-specific linguistic effects preferentially concentrate within long-wavelength cortical eigenmodes",
    fontsize=15, fontweight="bold", y=0.985,
)
plt.subplots_adjust(top=0.90, bottom=0.08, left=0.075, right=0.97)

png_file = FIGDIR / "figure3_validation_and_longwavelength_concentration.png"
pdf_file = FIGDIR / "figure3_validation_and_longwavelength_concentration.pdf"
fig.savefig(png_file, dpi=300, bbox_inches="tight")
fig.savefig(pdf_file, bbox_inches="tight")
plt.show()
print("\nSaved:")
print(png_file)
print(pdf_file)


## Figure 4 — Validated effects converge onto a low-dimensional eigenmode geometry
This section uses variance-standardized partial beta-star profiles. It therefore
tests cross-predictor **profile shape**, distinct from the raw-beta concentration
analysis in Figure 3.


In [ ]:
# ============================================================
# NEW FIGURE 4
# VALIDATED LINGUISTIC EFFECTS CONVERGE ONTO A LOW-DIMENSIONAL
# CORTICAL EIGENMODE GEOMETRY
#
# A. Variance-standardized partial beta* profiles
# B. Shape-PCA loading space
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = REPO_ROOT
PANG = BASE / "pang_out"
GEOM = PANG / "validated_linguistic_effects_geometry"
FIGDIR = PANG / "paper_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

profiles = pd.read_csv(GEOM / "validated_four_beta_star_profiles.csv", index_col=0)
profiles.index = profiles.index.astype(int)
weights = pd.read_csv(GEOM / "validated_four_shapePCA_component_weights.csv", index_col=0)
ev = pd.read_csv(GEOM / "validated_four_shapePCA_explained_variance.csv")
pc1_pct = float(ev.loc[ev["PC"] == "PC1", "explained_variance_percent"].iloc[0])
pc2_pct = float(ev.loc[ev["PC"] == "PC2", "explained_variance_percent"].iloc[0])
pc12_pct = pc1_pct + pc2_pct

COLORS = {
    "Sentence onset": "#0072B2",
    "Sentence shift": "#D55E00",
    "Token surprisal": "#009E73",
    "Token curvature": "#CC79A7",
}
PREDICTORS = ["Sentence onset", "Sentence shift", "Token surprisal", "Token curvature"]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13.8, 5.8), gridspec_kw={"width_ratios": [1.18, 1.0]})

# ------------------------- PANEL A -------------------------
for name in PREDICTORS:
    axA.plot(profiles.index, profiles[name], lw=2.15, color=COLORS[name], label=name)
axA.axhline(0, color="0.5", lw=0.8, ls="--", alpha=0.7)
axA.set_xlabel("Eigenmode index, $k$", fontsize=11)
axA.set_ylabel(r"Standardized partial $\beta^*$", fontsize=11)
axA.set_xlim(1, 119)
axA.legend(loc="upper center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=False, fontsize=8.5, handlelength=1.5, columnspacing=1.2)
axA.spines[["top", "right"]].set_visible(False)
axA.tick_params(labelsize=9.5)
axA.set_title("Variance-standardized eigenmode profiles", fontsize=12, fontweight="bold", pad=9)
axA.text(-0.12, 1.08, "A", transform=axA.transAxes, fontsize=18, fontweight="bold", va="top")

# ------------------------- PANEL B -------------------------
axB.axhline(0, color="0.75", lw=0.9)
axB.axvline(0, color="0.75", lw=0.9)
LABEL_OFFSETS = {
    "Sentence onset": (8, -8),
    "Sentence shift": (8, 5),
    "Token surprisal": (8, 8),
    "Token curvature": (-6, -13),
}
for name in PREDICTORS:
    x = float(weights.loc[name, "PC1"]); y = float(weights.loc[name, "PC2"])
    axB.annotate("", xy=(x, y), xytext=(0, 0),
                 arrowprops=dict(arrowstyle="-|>", lw=2.4, color=COLORS[name], mutation_scale=14))
    dx, dy = LABEL_OFFSETS[name]
    axB.annotate(name, xy=(x, y), xytext=(dx, dy), textcoords="offset points",
                 color=COLORS[name], fontsize=10, fontweight="bold",
                 ha="right" if dx < 0 else "left", va="center")

# Data-driven limits, retaining only a small negative-PC1 margin because all PC1 loadings are positive.
xvals = weights.loc[PREDICTORS, "PC1"].to_numpy(float)
yvals = weights.loc[PREDICTORS, "PC2"].to_numpy(float)
xmax = max(0.05, np.max(xvals) * 1.18)
yspan = max(abs(np.min(yvals)), abs(np.max(yvals))) * 1.30
axB.set_xlim(-0.04, xmax)
axB.set_ylim(-yspan, yspan)
axB.set_xlabel(f"PC1 ({pc1_pct:.1f}% variance)", fontsize=11)
axB.set_ylabel(f"PC2 ({pc2_pct:.1f}% variance)", fontsize=11)
axB.spines[["top", "right"]].set_visible(False)
axB.tick_params(labelsize=9.5)
axB.set_title(f"Shape-PCA loading space (PC1 + PC2 = {pc12_pct:.1f}%)",
              fontsize=12, fontweight="bold", pad=9)
axB.text(-0.12, 1.08, "B", transform=axB.transAxes, fontsize=18, fontweight="bold", va="top")

fig.suptitle("Validated linguistic effects converge onto a low-dimensional cortical eigenmode geometry",
             fontsize=15, fontweight="bold", y=0.985)
plt.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97, wspace=0.34)

png_file = FIGDIR / "figure4_validated_effects_lowdimensional_geometry.png"
pdf_file = FIGDIR / "figure4_validated_effects_lowdimensional_geometry.pdf"
fig.savefig(png_file, dpi=300, bbox_inches="tight")
fig.savefig(pdf_file, bbox_inches="tight")
plt.show()
print("\nSaved:")
print(png_file)
print(pdf_file)


## Figure 5 — K=20 cortical reconstruction of validated effects
The final reconstruction is restricted to the four validated linguistic effects and
shows empirical, K=20 reconstructed, and residual cortical maps.


In [ ]:
# ============================================================
# FINAL FIGURE 5
#
# Low-order cortical eigenmodes preserve large-scale spatial
# organization of validated linguistic responses
#
# Rows:
#   Sentence onset
#   Sentence shift
#   Token surprisal
#   Token curvature
#
# Columns:
#   Empirical partial beta
#   First 20 eigenmodes
#   Residual (empirical - K20)
#
# Each panel contains left and right lateral cortical surfaces.
#
# IMPORTANT:
#   - Same color scale across empirical / K20 / residual
#     WITHIN each effect.
#   - Scale determined from the 99th percentile of the
#     empirical partial-beta map.
#   - K20 R² shown separately for L and R hemispheres.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from nilearn import datasets, plotting


# ============================================================
# PATHS
# ============================================================

BASE = REPO_ROOT

RECON_ROOT = (
    BASE
    / "pang_out"
    / "validated_effects_reconstruction"
)

FIG_DIR = (
    BASE
    / "pang_out"
    / "paper_figures"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# EFFECTS
# ============================================================

EFFECTS = [
    ("sentence_onset",   "Sentence onset"),
    ("sentence_shift",   "Sentence shift"),
    ("token_surprisal",  "Token surprisal"),
    ("token_curvature",  "Token curvature"),
]

MAP_TYPES = [
    ("empirical", "Empirical partial β"),
    ("low",       "First 20 eigenmodes"),
    ("residual",  "Residual"),
]


# ============================================================
# RECONSTRUCTION STATISTICS
# ============================================================

stats_file = (
    RECON_ROOT
    / "validated_four_effect_reconstruction_diagnostics.csv"
)

stats = pd.read_csv(
    stats_file
)

r2_lookup = {}

for _, row in stats.iterrows():

    r2_lookup[
        (
            row["target"],
            row["hemisphere"],
        )
    ] = row[
        "R2_empirical_K20"
    ]


# ============================================================
# SURFACES
# ============================================================

fsavg = datasets.fetch_surf_fsaverage(
    mesh="fsaverage5"
)


# ============================================================
# LOAD MAPS
# ============================================================

maps = {}

for effect, _ in EFFECTS:

    maps[effect] = {}

    for map_type, _ in MAP_TYPES:

        maps[effect][map_type] = {}

        for hemi in ["L", "R"]:

            f = (
                RECON_ROOT
                / effect
                / (
                    f"{effect}_hemi-{hemi}_"
                    f"{map_type}.npy"
                )
            )

            maps[effect][map_type][hemi] = (
                np.load(f)
            )


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(15.5, 10.5)
)

gs = fig.add_gridspec(
    nrows=4,
    ncols=8,
    width_ratios=[
        0.72,       # row label
        1, 1,       # empirical L / R
        1, 1,       # K20 L / R
        1, 1,       # residual L / R
        0.055,      # colorbar
    ],
    wspace=0.01,
    hspace=0.02,
)


# ============================================================
# COLUMN HEADERS
# ============================================================

fig.text(
    0.255,
    0.935,
    "Empirical partial β",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
)

fig.text(
    0.510,
    0.935,
    "First 20 eigenmodes",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
)

fig.text(
    0.765,
    0.935,
    "Residual",
    ha="center",
    va="bottom",
    fontsize=12,
    fontweight="bold",
)


# ============================================================
# HEMISPHERE HEADERS
# ============================================================

hemi_positions = [
    (0.190, "L"),
    (0.318, "R"),
    (0.445, "L"),
    (0.573, "R"),
    (0.700, "L"),
    (0.828, "R"),
]

for x, label in hemi_positions:

    fig.text(
        x,
        0.918,
        label,
        ha="center",
        va="center",
        fontsize=9,
    )


# ============================================================
# PLOT EACH EFFECT
# ============================================================

for row, (effect, label) in enumerate(EFFECTS):

    # --------------------------------------------------------
    # Common scale within effect.
    #
    # Defined entirely from empirical partial-beta maps.
    # --------------------------------------------------------

    empirical_values = np.concatenate([
        maps[effect]["empirical"]["L"][
            np.isfinite(
                maps[effect]["empirical"]["L"]
            )
        ],
        maps[effect]["empirical"]["R"][
            np.isfinite(
                maps[effect]["empirical"]["R"]
            )
        ],
    ])

    vmax = np.percentile(
        np.abs(empirical_values),
        99,
    )


    # --------------------------------------------------------
    # ROW LABEL
    # --------------------------------------------------------

    label_ax = fig.add_subplot(
        gs[row, 0]
    )

    label_ax.axis("off")

    label_ax.text(
        0.98,
        0.50,
        label,
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold",
    )


    # --------------------------------------------------------
    # COLUMN LOCATIONS
    # --------------------------------------------------------

    col_pairs = {
        "empirical": (1, 2),
        "low":       (3, 4),
        "residual":  (5, 6),
    }


    # --------------------------------------------------------
    # PLOT MAPS
    # --------------------------------------------------------

    for map_type, _ in MAP_TYPES:

        cL, cR = col_pairs[
            map_type
        ]


        # LEFT HEMISPHERE
        axL = fig.add_subplot(
            gs[row, cL],
            projection="3d",
        )

        plotting.plot_surf_stat_map(
            surf_mesh=
                fsavg.pial_left,

            stat_map=
                maps[effect][map_type]["L"],

            bg_map=
                fsavg.sulc_left,

            hemi="left",
            view="lateral",
            axes=axL,

            colorbar=False,
            cmap="RdBu_r",
            symmetric_cbar=True,
            vmax=vmax,
            threshold=None,
            bg_on_data=True,
            darkness=None,
        )


        # RIGHT HEMISPHERE
        axR = fig.add_subplot(
            gs[row, cR],
            projection="3d",
        )

        plotting.plot_surf_stat_map(
            surf_mesh=
                fsavg.pial_right,

            stat_map=
                maps[effect][map_type]["R"],

            bg_map=
                fsavg.sulc_right,

            hemi="right",
            view="lateral",
            axes=axR,

            colorbar=False,
            cmap="RdBu_r",
            symmetric_cbar=True,
            vmax=vmax,
            threshold=None,
            bg_on_data=True,
            darkness=None,
        )


        # ----------------------------------------------------
        # R² LABELS ONLY FOR K20
        # ----------------------------------------------------

        if map_type == "low":

            r2_L = r2_lookup[
                (effect, "L")
            ]

            r2_R = r2_lookup[
                (effect, "R")
            ]

            axL.text2D(
                0.50,
                0.04,
                rf"$R^2$ = {r2_L:.2f}",
                transform=axL.transAxes,
                ha="center",
                va="bottom",
                fontsize=8.5,
            )

            axR.text2D(
                0.50,
                0.04,
                rf"$R^2$ = {r2_R:.2f}",
                transform=axR.transAxes,
                ha="center",
                va="bottom",
                fontsize=8.5,
            )


    # --------------------------------------------------------
    # COLORBAR
    # --------------------------------------------------------

    cax = fig.add_subplot(
        gs[row, 7]
    )

    norm = mpl.colors.Normalize(
        vmin=-vmax,
        vmax=vmax,
    )

    sm = mpl.cm.ScalarMappable(
        norm=norm,
        cmap="RdBu_r",
    )

    sm.set_array([])

    cb = fig.colorbar(
        sm,
        cax=cax,
    )

    cb.ax.tick_params(
        labelsize=7
    )

    cb.set_label(
        "partial β",
        fontsize=8,
    )


# ============================================================
# TITLE
# ============================================================

fig.suptitle(
    "Low-order cortical eigenmodes preserve the large-scale "
    "organization of linguistic responses",
    fontsize=14,
    y=0.985,
)


# ============================================================
# SAVE
# ============================================================

png_file = (
    FIG_DIR
    / "figure5_validated_effects_K20_reconstruction_FINAL.png"
)

pdf_file = (
    FIG_DIR
    / "figure5_validated_effects_K20_reconstruction_FINAL.pdf"
)


fig.savefig(
    png_file,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    pdf_file,
    bbox_inches="tight",
    facecolor="white",
)


print(
    f"Saved: {png_file}"
)

print(
    f"Saved: {pdf_file}"
)

plt.show()


## Figure 6 — Subcortical and cortico-subcortical coordination
The definitive code reloads authoritative participant-level outputs under
`pang_out/subcortex/validated_effects_20260921/`; it does not recompute the upstream
hippocampal/VTA GLMs here.


In [ ]:
# ============================================================
# FIGURE 6 — DEFINITIVE SUBCORTICAL / CORTICO-SUBCORTICAL FIGURE
#
# A. Hippocampal mean-signal responses
# B. Hippocampal spatial-mode responses
# C. VTA mean-signal responses
# D. Baseline VTA–hippocampal coupling
# E. Sentence modulation of VTA–hippocampal cofluctuation
# F. Sentence modulation of VTA–cortical modal cofluctuation
#
# All panels load the authoritative participant-level outputs
# generated earlier in:
# 05_subcortex_validated_effects_20260921.ipynb
#
# No GLMs are recomputed here.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# 1. PATHS
# ============================================================

BASE = REPO_ROOT
PANG = BASE / "pang_out"

OUT_ROOT = (
    PANG
    / "subcortex"
    / "validated_effects_20260921"
)

HC_MEAN_FILE = (
    OUT_ROOT
    / "hippocampal_mean_signal"
    / "hipp_mean_validated_effects_participant_bilateral.csv"
)

HC_EIGEN_FILE = (
    OUT_ROOT
    / "hippocampal_eigenspace_language"
    / "hc_eigenspace_bilateral_participant.csv"
)

VTA_MEAN_FILE = (
    OUT_ROOT
    / "vta_mean_signal"
    / "vta_bilateral_participant.csv"
)

VTA_HC_COUPLING_FILE = (
    OUT_ROOT
    / "vta_hipp_coupling"
    / "vta_hc_coupling_subject_bilateral.csv"
)

VTA_HC_COFLUCT_FILE = (
    OUT_ROOT
    / "vta_hipp_sentence_cofluctuation"
    / "vta_hc_cofluctuation_bilateral_participant.csv"
)

VTA_CORTICAL_FILE = (
    OUT_ROOT
    / "vta_cortical_sentence_cofluctuation"
    / "vta_cortical_sentence_cofluctuation_participant_bilateral.csv"
)

LOO_FILE = (
    OUT_ROOT
    / "vta_cortical_sentence_cofluctuation"
    / "sentence_modulation_LOO_paired_participants.csv"
)

LOO_INFERENCE_FILE = (
    OUT_ROOT
    / "vta_cortical_sentence_cofluctuation"
    / "sentence_modulation_LOO_inference.csv"
)

FIG_OUT = OUT_ROOT / "figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. LOAD
# ============================================================

hc_mean = pd.read_csv(HC_MEAN_FILE)
hc_eigen = pd.read_csv(HC_EIGEN_FILE)
vta_mean = pd.read_csv(VTA_MEAN_FILE)
vta_hc_coupling = pd.read_csv(VTA_HC_COUPLING_FILE)
vta_hc_cofluct = pd.read_csv(VTA_HC_COFLUCT_FILE)
vta_cortical = pd.read_csv(VTA_CORTICAL_FILE)
loo = pd.read_csv(LOO_FILE)
loo_inf = pd.read_csv(LOO_INFERENCE_FILE)


# ============================================================
# 3. EXACT SCHEMA CHECKS
# ============================================================

EXPECTED = {
    "HC mean": (
        hc_mean,
        {
            "subject",
            "effect",
            "beta",
            "beta_star",
            "n_hemispheres",
            "mean_n_runs",
        },
    ),

    "HC eigenspace": (
        hc_eigen,
        {
            "sub",
            "outcome",
            "effect",
            "beta",
            "beta_star",
            "n_hemis",
        },
    ),

    "VTA mean": (
        vta_mean,
        {
            "sub",
            "effect",
            "beta",
            "beta_star",
            "n_hemis",
        },
    ),

    "VTA-HC coupling": (
        vta_hc_coupling,
        {
            "subject",
            "z_raw",
            "z_global_resid",
            "z_diff_global_resid",
            "n_pairings",
        },
    ),

    "VTA-HC cofluctuation": (
        vta_hc_cofluct,
        {
            "subject",
            "predictor",
            "beta_star",
            "beta",
            "n_pairings",
        },
    ),

    "VTA-cortical": (
        vta_cortical,
        {
            "subject",
            "mode_k",
            "predictor",
            "beta_star",
            "beta",
            "n_pairings",
        },
    ),

    "LOO": (
        loo,
        {
            "subject",
            "sentence_onset",
            "sentence_shift",
        },
    ),

    "LOO inference": (
        loo_inf,
        {
            "contrast",
            "n",
            "mean_fisher_z",
            "ci_low",
            "ci_high",
            "cohens_dz",
            "t",
            "df",
            "p_two_sided",
            "p_one_sided",
        },
    ),
}


print("=" * 70)
print("FIGURE 6 INPUT CHECK")
print("=" * 70)

for name, (df, cols) in EXPECTED.items():

    missing = cols - set(df.columns)

    if missing:
        raise RuntimeError(
            f"{name}: missing columns {missing}"
        )

    print(
        f"{name:25s} "
        f"{df.shape}"
    )


# ============================================================
# 4. DEFINITIONS
# ============================================================

EFFECT_ORDER = [
    "sentence_onset",
    "sentence_shift",
    "surprisal",
    "curvature",
]

EFFECT_LABELS = {
    "sentence_onset": "Sentence\nonset",
    "sentence_shift": "Sentence\nshift",
    "surprisal": "Surprisal",
    "curvature": "Curvature",
}

EFFECT_COLORS = {
    "sentence_onset": "#0072B2",
    "sentence_shift": "#E69F00",
    "surprisal": "#009E73",
    "curvature": "#CC79A7",
}


# ============================================================
# 5. HELPERS
# ============================================================

def mean_ci(x):
    """
    Participant-level mean and 95% t confidence interval.
    """

    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    n = len(x)
    mean = np.mean(x)
    sd = np.std(x, ddof=1)
    sem = sd / np.sqrt(n)

    crit = stats.t.ppf(
        0.975,
        df=n - 1
    )

    return {
        "n": n,
        "mean": mean,
        "sd": sd,
        "sem": sem,
        "ci_low": mean - crit * sem,
        "ci_high": mean + crit * sem,
    }


def clean_axis(ax, zero=True):

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        direction="out",
        length=3,
        width=0.8,
    )

    if zero:
        ax.axhline(
            0,
            color="0.55",
            linestyle="--",
            linewidth=0.75,
            zorder=0,
        )


def panel_letter(ax, letter):

    ax.text(
        -0.14,
        1.08,
        letter,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="top",
    )


def plot_effect_scatter(
    ax,
    df,
    effect_col,
    value_col,
    effects,
    labels,
    ylabel,
    title,
    seed,
):

    rng = np.random.default_rng(seed)

    for i, effect in enumerate(effects):

        vals = (
            df.loc[
                df[effect_col] == effect,
                value_col,
            ]
            .dropna()
            .to_numpy(dtype=float)
        )

        jitter = rng.uniform(
            -0.09,
            0.09,
            size=len(vals)
        )

        ax.scatter(
            i + jitter,
            vals,
            s=14,
            alpha=0.25,
            color=EFFECT_COLORS[effect],
            edgecolors="none",
            zorder=1,
        )

        s = mean_ci(vals)

        ax.errorbar(
            i,
            s["mean"],
            yerr=np.array([
                [s["mean"] - s["ci_low"]],
                [s["ci_high"] - s["mean"]],
            ]),
            fmt="o",
            markersize=6,
            capsize=3,
            linewidth=1.5,
            color=EFFECT_COLORS[effect],
            markeredgecolor="black",
            markeredgewidth=0.4,
            zorder=4,
        )

    ax.set_xticks(
        np.arange(len(effects))
    )

    ax.set_xticklabels(
        [labels[e] for e in effects]
    )

    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=7)

    clean_axis(ax)


# ============================================================
# 6. SANITY CHECK FINAL EFFECT MEANS
# ============================================================

print("\n" + "=" * 70)
print("GROUP MEANS USED IN PANELS A, C, E")
print("=" * 70)


print("\nA. HC mean")
print(
    hc_mean.groupby("effect")[
        "beta_star"
    ].agg(["count", "mean", "std"])
)


print("\nC. VTA mean")
print(
    vta_mean.groupby("effect")[
        "beta_star"
    ].agg(["count", "mean", "std"])
)


print("\nE. VTA-HC cofluctuation")
print(
    vta_hc_cofluct.groupby("predictor")[
        "beta_star"
    ].agg(["count", "mean", "std"])
)


print("\nD. Baseline VTA-HC coupling")
print(
    vta_hc_coupling[
        [
            "z_global_resid",
            "z_diff_global_resid",
        ]
    ].agg(
        ["count", "mean", "std"]
    )
)


# ============================================================
# 7. PREPARE PANEL B
# ============================================================

HC_OUTCOMES = [
    "mode1",
    "mode2",
    "mode3",
    "higher_4_29",
]

HC_OUTCOME_LABELS = {
    "mode1": "Mode 1",
    "mode2": "Mode 2",
    "mode3": "Mode 3",
    "higher_4_29": "Modes\n4–29",
}


hc_eigen_plot = (
    hc_eigen[
        hc_eigen["outcome"].isin(HC_OUTCOMES)
        &
        hc_eigen["effect"].isin(EFFECT_ORDER)
    ]
    .copy()
)


# ============================================================
# 8. PREPARE PANEL F
# ============================================================

F_EFFECTS = [
    "sentence_onset",
    "sentence_shift",
]


f_rows = []

for effect in F_EFFECTS:

    de = vta_cortical[
        vta_cortical["predictor"] == effect
    ]

    for mode_k, dm in de.groupby("mode_k"):

        s = mean_ci(
            dm["beta_star"].to_numpy()
        )

        f_rows.append({
            "predictor": effect,
            "mode_k": int(mode_k),
            **s,
        })


f_group = pd.DataFrame(f_rows)


# Exact LOO inference result.
loo_diff = (
    loo_inf.loc[
        loo_inf["contrast"]
        == "sentence_onset_minus_shift"
    ]
)

if len(loo_diff) != 1:
    raise RuntimeError(
        "Expected exactly one "
        "sentence_onset_minus_shift row."
    )

loo_diff = loo_diff.iloc[0]


# ============================================================
# 9. TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8.5,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# 10. FIGURE LAYOUT
#
# Revised:
# - substantially reduced vertical gap between rows
# - Panel F inset removed
# - F retains full width for the 119-mode profile
# ============================================================

fig = plt.figure(
    figsize=(12.4, 6.35)
)

gs = fig.add_gridspec(
    2,
    12,
    left=0.065,
    right=0.985,
    bottom=0.10,
    top=0.95,
    wspace=1.70,
    hspace=0.34,
)


axA = fig.add_subplot(gs[0, 0:3])
axB = fig.add_subplot(gs[0, 3:8])
axC = fig.add_subplot(gs[0, 8:12])

axD = fig.add_subplot(gs[1, 0:3])
axE = fig.add_subplot(gs[1, 3:6])
axF = fig.add_subplot(gs[1, 6:12])


# ============================================================
# 11. PANEL A
# HIPPOCAMPAL MEAN SIGNAL
# ============================================================

plot_effect_scatter(
    ax=axA,
    df=hc_mean,
    effect_col="effect",
    value_col="beta_star",
    effects=EFFECT_ORDER,
    labels=EFFECT_LABELS,
    ylabel=r"Variance-standardized coefficient ($\beta^*$)",
    title="Hippocampal mean signal",
    seed=20260921,
)

axA.set_ylim(
    -0.11,
    0.04
)

panel_letter(
    axA,
    "A"
)


# ============================================================
# 12. PANEL B
# HIPPOCAMPAL SPATIAL MODES
# ============================================================

xB = np.arange(
    len(HC_OUTCOMES)
)


for effect in EFFECT_ORDER:

    means = []
    lows = []
    highs = []

    for outcome in HC_OUTCOMES:

        vals = (
            hc_eigen_plot.loc[
                (
                    hc_eigen_plot["effect"]
                    == effect
                )
                &
                (
                    hc_eigen_plot["outcome"]
                    == outcome
                ),
                "beta_star",
            ]
            .dropna()
            .to_numpy(dtype=float)
        )

        s = mean_ci(vals)

        means.append(s["mean"])
        lows.append(s["ci_low"])
        highs.append(s["ci_high"])

    means = np.asarray(means)
    lows = np.asarray(lows)
    highs = np.asarray(highs)

    axB.fill_between(
        xB,
        lows,
        highs,
        color=EFFECT_COLORS[effect],
        alpha=0.12,
        linewidth=0,
        zorder=1,
    )

    axB.plot(
        xB,
        means,
        marker="o",
        markersize=4.5,
        linewidth=1.5,
        color=EFFECT_COLORS[effect],
        label=EFFECT_LABELS[
            effect
        ].replace("\n", " "),
        zorder=3,
    )


# Visual separation between individually displayed
# low-order modes and the aggregate higher-mode band.
axB.axvline(
    2.5,
    color="0.70",
    linestyle=":",
    linewidth=0.8,
)


axB.set_xticks(xB)

axB.set_xticklabels([
    HC_OUTCOME_LABELS[x]
    for x in HC_OUTCOMES
])

axB.set_ylabel(
    r"Variance-standardized coefficient ($\beta^*$)"
)

axB.set_title(
    "Hippocampal spatial modes",
    pad=7
)

axB.set_ylim(
    -0.11,
    0.04
)

clean_axis(axB)


axB.legend(
    frameon=False,
    ncol=2,
    loc="lower right",
    handlelength=1.6,
    columnspacing=0.9,
)

panel_letter(
    axB,
    "B"
)


# ============================================================
# 13. PANEL C
# VTA MEAN SIGNAL
# ============================================================

plot_effect_scatter(
    ax=axC,
    df=vta_mean,
    effect_col="effect",
    value_col="beta_star",
    effects=EFFECT_ORDER,
    labels=EFFECT_LABELS,
    ylabel=r"Variance-standardized coefficient ($\beta^*$)",
    title="VTA mean signal",
    seed=20260922,
)

axC.set_ylim(
    -0.11,
    0.04
)

panel_letter(
    axC,
    "C"
)


# ============================================================
# 14. PANEL D
# BASELINE VTA-HIPPOCAMPAL COUPLING
#
# Raw coupling deliberately omitted.
# ============================================================

D_VARIABLES = [
    "z_global_resid",
    "z_diff_global_resid",
]

D_LABELS = [
    "Global-\nresidualized",
    "First-\ndifference",
]

rngD = np.random.default_rng(
    20260923
)


for i, variable in enumerate(
    D_VARIABLES
):

    vals = (
        vta_hc_coupling[
            variable
        ]
        .dropna()
        .to_numpy(dtype=float)
    )

    jitter = rngD.uniform(
        -0.075,
        0.075,
        size=len(vals)
    )

    axD.scatter(
        i + jitter,
        vals,
        s=14,
        alpha=0.25,
        color="0.35",
        edgecolors="none",
        zorder=1,
    )

    s = mean_ci(vals)

    axD.errorbar(
        i,
        s["mean"],
        yerr=np.array([
            [
                s["mean"]
                - s["ci_low"]
            ],
            [
                s["ci_high"]
                - s["mean"]
            ],
        ]),
        fmt="o",
        markersize=6,
        capsize=3,
        linewidth=1.5,
        color="black",
        zorder=4,
    )


axD.set_xticks(
    [0, 1]
)

axD.set_xticklabels(
    D_LABELS
)

axD.set_ylabel(
    "VTA–hippocampal coupling\n"
    "(Fisher $z$)"
)

axD.set_title(
    "Baseline VTA–hippocampal coupling",
    pad=7
)

clean_axis(axD)

panel_letter(
    axD,
    "D"
)


# ============================================================
# 15. PANEL E
# SENTENCE MODULATION OF VTA-HC COFLUCTUATION
# ============================================================

E_EFFECTS = [
    "sentence_onset",
    "sentence_shift",
]

E_LABELS = {
    "sentence_onset": "Sentence\nonset",
    "sentence_shift": "Sentence\nshift",
}


plot_effect_scatter(
    ax=axE,
    df=vta_hc_cofluct,
    effect_col="predictor",
    value_col="beta_star",
    effects=E_EFFECTS,
    labels=E_LABELS,
    ylabel=r"Cofluctuation modulation ($\beta^*$)",
    title=(
        "Sentence modulation of\n"
        "VTA–hippocampal cofluctuation"
    ),
    seed=20260924,
)

panel_letter(
    axE,
    "E"
)


# ============================================================
# 16. PANEL F
# SENTENCE MODULATION OF VTA-CORTICAL MODAL COFLUCTUATION
#
# Main object:
# mode-specific modulation profile across 119 non-global
# cortical eigenmodes.
#
# Participant-based 95% CIs are shown.
#
# Cross-participant LOO reproducibility is NOT shown inside
# the panel; those statistics belong in the figure caption.
# ============================================================

for effect in F_EFFECTS:

    d = (
        f_group[
            f_group["predictor"]
            == effect
        ]
        .sort_values("mode_k")
    )

    x = d["mode_k"].to_numpy()
    y = d["mean"].to_numpy()
    lo = d["ci_low"].to_numpy()
    hi = d["ci_high"].to_numpy()

    axF.fill_between(
        x,
        lo,
        hi,
        color=EFFECT_COLORS[effect],
        alpha=0.12,
        linewidth=0,
        zorder=1,
    )

    axF.plot(
        x,
        y,
        color=EFFECT_COLORS[effect],
        linewidth=1.45,
        label=(
            "Sentence onset"
            if effect == "sentence_onset"
            else "Sentence shift"
        ),
        zorder=3,
    )


axF.set_xlim(
    1,
    119
)

axF.set_xticks([
    1,
    20,
    40,
    60,
    80,
    100,
    119,
])

axF.set_xlabel(
    "Cortical eigenmode"
)

axF.set_ylabel(
    r"Cofluctuation modulation ($\beta^*$)"
)

axF.set_title(
    "Sentence modulation of VTA–cortical modal cofluctuation",
    pad=7
)

clean_axis(axF)


axF.legend(
    frameon=False,
    loc="upper right",
    handlelength=1.8,
)

panel_letter(
    axF,
    "F"
)


# ============================================================
# 17. SAVE
# ============================================================

PNG_FILE = (
    FIG_OUT
    / "Figure6_subcortical_validated_effects.png"
)

PDF_FILE = (
    FIG_OUT
    / "Figure6_subcortical_validated_effects.pdf"
)

SVG_FILE = (
    FIG_OUT
    / "Figure6_subcortical_validated_effects.svg"
)


fig.savefig(
    PNG_FILE,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    PDF_FILE,
    bbox_inches="tight",
    facecolor="white",
)

fig.savefig(
    SVG_FILE,
    bbox_inches="tight",
    facecolor="white",
)


print("\n" + "=" * 70)
print("FIGURE 6 SAVED")
print("=" * 70)

print(PNG_FILE)
print(PDF_FILE)
print(SVG_FILE)


plt.show()


## Reproducibility boundary
Successful execution reproduces Figures 2–6 and key numerical summaries from final
precomputed outputs. It is not a claim that one notebook reprocesses the raw
OpenNeuro dataset end-to-end. Upstream clean notebooks/scripts document production
of the precomputed outputs.

The notebook deliberately avoids “first existing file” logic: canonical inputs must
exist at their documented paths.
